In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score

# 1. Cargar y ordenar
df_hmm_ready = pd.read_csv('../data/processed/hmm.csv')
df_hmm_ready = df_hmm_ready.sort_values(['animal_id', 'trayectoria_id', 'date'])

# 2. Crear target por segmento y filtrar NaNs
df_hmm_ready['target_cell'] = df_hmm_ready.groupby('trayectoria_id')['cell_id'].shift(-1)
df_hmm_ready_filtered = df_hmm_ready.dropna(subset=['target_cell']).copy()

In [2]:
# --- NUEVA CELDA 2 (SÓLO SEMANA) ---
import pandas as pd
import numpy as np

# 1. Aseguramos formato datetime
df_hmm_ready['date'] = pd.to_datetime(df_hmm_ready['date'])

# 2. Única variable temporal: Semana del año (1 a 52)
# Eliminamos mes_num para evitar redundancia
df_hmm_ready['semana_num'] = df_hmm_ready['date'].dt.isocalendar().week.astype(int)

# 3. Filtramos para crear el dataset de entrenamiento
df_hmm_ready_filtered = df_hmm_ready.dropna(subset=['target_cell']).copy()

# 4. Definir lista de features (SIN MES, SIN LUNA, SIN DÍA)
features = [
    'grid_x', 'grid_y', 'step_length', 'turning_angle', 'bearing', 
    'estado_hmm', 'veg_low', 'veg_high', 
    'semana_num' # Única referencia temporal
]

X = df_hmm_ready_filtered[features]
y_all_raw = df_hmm_ready_filtered['target_cell']

print(f"Dataset preparado con {len(features)} variables.")
print(f"Features: {features}")

Dataset preparado con 9 variables.
Features: ['grid_x', 'grid_y', 'step_length', 'turning_angle', 'bearing', 'estado_hmm', 'veg_low', 'veg_high', 'semana_num']


In [3]:
# Sustituye tu bloque #4 por este:

X_train_list, X_test_list = [], []
y_train_list, y_test_list = [], []

# Agrupamos por animal para asegurar que cada uno tenga su 80% de entrenamiento y 20% de test
for animal, df_animal in df_hmm_ready_filtered.groupby('animal_id'):
    # Ordenar por fecha por si acaso
    df_animal = df_animal.sort_values('date')
    
    n = len(df_animal)
    if n < 5: continue # Omitir aves con poquísimos datos si las hubiera
    
    split_idx = int(n * 0.8)
    
    # Dividir datos del ave
    X_train_list.append(df_animal[features].iloc[:split_idx])
    X_test_list.append(df_animal[features].iloc[split_idx:])
    y_train_list.append(df_animal['target_cell'].iloc[:split_idx])
    y_test_list.append(df_animal['target_cell'].iloc[split_idx:])

# Concatenar todos los resultados
X_train_raw = pd.concat(X_train_list)
X_test_raw = pd.concat(X_test_list)
y_train_raw = pd.concat(y_train_list)
y_test_raw = pd.concat(y_test_list)

print(f"Dataset listo. Registros Train: {len(X_train_raw)}, Registros Test: {len(X_test_raw)}")

Dataset listo. Registros Train: 16323, Registros Test: 4137


In [4]:

# 5. LabelEncoder basado SOLO en entrenamiento
le_final = LabelEncoder()
y_train = le_final.fit_transform(y_train_raw)

# 6. Filtrar Test para que solo tenga celdas vistas en Train
mask_test = y_test_raw.isin(le_final.classes_)
X_test = X_test_raw[mask_test].copy() # .copy() para evitar warnings de SettingWithCopy
y_test = le_final.transform(y_test_raw[mask_test])

X_train = X_train_raw

In [5]:
# 7. Modelos
#Diccionario de modelos ACTUALIZADO para evitar el 0% de Accuracy
num_clases = len(le_final.classes_)

modelos = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(
        n_estimators=100, 
        learning_rate=0.1, 
        random_state=42, 
        objective='multi:softprob', 
        num_class=num_clases,
        eval_metric='mlogloss'
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=200,          # Aumentamos árboles para captar más detalle
        learning_rate=0.05,        # Aprendizaje más lento y preciso
        num_leaves=31,             # Complejidad del árbol estándar
        min_child_samples=5,       # CLAVE: Permite crear hojas con pocos datos (importante para celdas poco visitadas)
        objective='multiclass',    # Forzamos clasificación multiclase
        num_class=num_clases,      # Número total de celdas
        random_state=42,
        verbose=-1,
        force_col_wise=True
    )
}

# 8. Bucle de entrenamiento
resultados = []
for nombre, model in modelos.items():
    inicio = time.time()
    print(f"Entrenando {nombre}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    resultados.append({"Modelo": nombre, "Accuracy": acc, "Tiempo (s)": time.time() - inicio})
    print(f"✅ {nombre} finalizado. Accuracy: {acc:.4f}")

print("\n--- COMPARATIVA FINAL (POR SEGMENTOS) ---")
print(pd.DataFrame(resultados).sort_values(by="Accuracy", ascending=False))

Entrenando Random Forest...


✅ Random Forest finalizado. Accuracy: 0.8114
Entrenando XGBoost...


✅ XGBoost finalizado. Accuracy: 0.8109
Entrenando LightGBM...


✅ LightGBM finalizado. Accuracy: 0.4287

--- COMPARATIVA FINAL (POR SEGMENTOS) ---
          Modelo  Accuracy  Tiempo (s)
0  Random Forest  0.811421    4.866811
1        XGBoost  0.810890   31.919897
2       LightGBM  0.428685   56.670796


In [6]:
import numpy as np

# 1. Usamos el Random Forest que acaba de dar el 81.14%
rf_definitivo = modelos["Random Forest"]
y_probs = rf_definitivo.predict_proba(X_test)

# 2. Calculamos Top-3 y Top-5
# np.argsort nos da los índices ordenados por probabilidad
top3_acc = np.mean(np.any(np.argsort(y_probs, axis=1)[:, -3:] == np.array(y_test).reshape(-1, 1), axis=1))
top5_acc = np.mean(np.any(np.argsort(y_probs, axis=1)[:, -5:] == np.array(y_test).reshape(-1, 1), axis=1))

print("--- EVALUACIÓN FINAL DE PRECISIÓN ---")
print(f"1. Accuracy Exacto (Top-1): {0.8114}")
print(f"2. Accuracy de Vecindario (Top-3): {top3_acc:.4f}")
print(f"3. Accuracy de Área (Top-5):      {top5_acc:.4f}")

if top3_acc >= 0.90 or top5_acc >= 0.90:
    print(f"\n🎉 ¡OBJETIVO CONSEGUIDO! El modelo tiene una fiabilidad >90% en el área predicha.")

--- EVALUACIÓN FINAL DE PRECISIÓN ---
1. Accuracy Exacto (Top-1): 0.8114
2. Accuracy de Vecindario (Top-3): 0.8821
3. Accuracy de Área (Top-5):      0.8967


In [7]:
# Accuracy breakdown by HMM state (migration vs stationary)
mask_migration  = X_test["estado_hmm"] == 0
mask_stationary = X_test["estado_hmm"] == 1

y_pred_all = rf_definitivo.predict(X_test)

acc_mig  = accuracy_score(y_test[mask_migration],  y_pred_all[mask_migration])
acc_stat = accuracy_score(y_test[mask_stationary], y_pred_all[mask_stationary])

# Top-3 and Top-5 per state
def topk_acc(probs, labels, mask, k):
    top_k = np.argsort(probs[mask], axis=1)[:, -k:]
    return np.mean(np.any(top_k == labels[mask].reshape(-1, 1), axis=1))

y_probs_all = rf_definitivo.predict_proba(X_test)
y_test_arr  = np.array(y_test)
mask_mig_arr  = mask_migration.values
mask_stat_arr = mask_stationary.values

print("--- ACCURACY BY HMM STATE ---")
print(f"Migration  (state 0): n={mask_mig_arr.sum():4d}  Top-1={acc_mig:.4f}  Top-3={topk_acc(y_probs_all, y_test_arr, mask_mig_arr, 3):.4f}  Top-5={topk_acc(y_probs_all, y_test_arr, mask_mig_arr, 5):.4f}")
print(f"Stationary (state 1): n={mask_stat_arr.sum():4d}  Top-1={acc_stat:.4f}  Top-3={topk_acc(y_probs_all, y_test_arr, mask_stat_arr, 3):.4f}  Top-5={topk_acc(y_probs_all, y_test_arr, mask_stat_arr, 5):.4f}")


--- ACCURACY BY HMM STATE ---
Migration  (state 0): n= 584  Top-1=0.4795  Top-3=0.6370  Top-5=0.7021
Stationary (state 1): n=3181  Top-1=0.8724  Top-3=0.9271  Top-5=0.9324
